# revT2V: zero-shot Attention Rotation test

**Question:** if we rotate ModelScope's temporal self-attention maps by 180° (no training at all), does the generated motion run backward?

For each prompt we generate 3 videos with the **same seed**:
- `normal`: plain teacher
- `rot_all`: rotation in all temporal attention layers (`attn1`)
- `rot_up`: rotation only in the U-Net up blocks (the paper's no-fine-tuning ablation)

Runtime: GPU (T4). About 1-2 min per video.

## 1. Setup

In [1]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# Change this to your GitHub username
GITHUB_USER = "silentlooop"  

!git clone -q https://github.com/{GITHUB_USER}/revT2V.git /content/revT2V 2>/dev/null || git -C /content/revT2V pull -q
%cd /content/revT2V
!pip install -q -e .
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## 2. Load the teacher

In [ ]:
import numpy as np
from diffusers.models.attention_processor import AttnProcessor, AttnProcessor2_0
from diffusers.utils import export_to_video
from IPython.display import Video, display

from revt2v.teacher import load_teacher
from revt2v.methods.attn_rotation import RotatedTemporalAttnProcessor   # adjust if your class lives elsewhere

pipe = load_teacher()
pipe.set_progress_bar_config(disable=False)

print("prediction_type:", pipe.scheduler.config.prediction_type)
print("text hidden size:", pipe.text_encoder.config.hidden_size)
print("unet frozen:", not any(p.requires_grad for p in pipe.unet.parameters()))

## 3. Find the temporal attention layers

In [ ]:
names = list(pipe.unet.attn_processors.keys())
temporal = [n for n in names if "temp_attentions" in n or "transformer_in" in n]

print(f"{len(names)} attention layers total, {len(temporal)} temporal")
for n in temporal:
    print("  ", n)

## 4. Sanity check: is the rotated processor correct?

On one real temporal layer with a random input, compare your processor against a reference: the stock attention with its map explicitly rotated 180°. They should match up to fp16 noise.

In [ ]:
attn_name = next(n for n in temporal if "temp_attentions" in n and "attn1" in n)
attn = pipe.unet.get_submodule(attn_name.rsplit(".processor", 1)[0])

C_in = attn.to_q.in_features
x = torch.randn(4, 16, C_in, device="cuda", dtype=pipe.unet.dtype)   # (pixels, frames, channels)

@torch.no_grad()
def reference_rotated(attn, h):
    q = attn.head_to_batch_dim(attn.to_q(h))
    k = attn.head_to_batch_dim(attn.to_k(h))
    v = attn.head_to_batch_dim(attn.to_v(h))
    probs = attn.get_attention_scores(q, k, None)      # (pixels*heads, 16, 16)
    probs = probs.flip(-1).flip(-2)                     # rotate 180°
    out = attn.batch_to_head_dim(torch.bmm(probs, v))
    out = attn.to_out[1](attn.to_out[0](out))
    if attn.residual_connection:
        out = out + h
    return out / attn.rescale_output_factor

with torch.no_grad():
    ours = RotatedTemporalAttnProcessor()(attn, x)
    ref = reference_rotated(attn, x)
    stock = AttnProcessor()(attn, x)

print("max |ours - reference| :", (ours - ref).abs().max().item(), " <- should be ~1e-3 or smaller")
print("max |ours - stock|     :", (ours - stock).abs().max().item(), " <- should be clearly larger (rotation changes things)")

## 5. Helpers

In [ ]:
def reset():
    pipe.unet.set_attn_processor(AttnProcessor2_0())

def rotate(filter_fn):
    procs = {n: (RotatedTemporalAttnProcessor() if filter_fn(n) else AttnProcessor2_0())
             for n in pipe.unet.attn_processors}
    pipe.unet.set_attn_processor(procs)
    k = sum(isinstance(p, RotatedTemporalAttnProcessor) for p in pipe.unet.attn_processors.values())
    print(f"  rotated {k} layers")

@torch.no_grad()
def gen(prompt, seed=0, steps=25):
    g = torch.Generator("cuda").manual_seed(seed)
    return pipe(prompt, num_frames=16, height=256, width=256,
                num_inference_steps=steps, generator=g, output_type="np").frames[0]   # (16, 256, 256, 3)

def side_by_side(videos):
    return np.concatenate(videos, axis=2)   # stack along width -> (16, 256, 256*len, 3)

VARIANTS = {
    "normal":  None,
    "rot_all": lambda n: "temp_attentions" in n and "attn1" in n,
    "rot_up":  lambda n: "up_blocks" in n and "temp_attentions" in n and "attn1" in n,
}

## 6. Run the experiment

In [ ]:
import os
os.makedirs("results/zero_shot_rotation", exist_ok=True)

prompts = [
    "a glass falling off a table and shattering on the floor",
    "a person walking forward on a beach",
    "a red car driving to the right on a road",
    "a candle burning down",
]
SEED = 0

for i, p in enumerate(prompts):
    print(f"\n[{i}] {p}")
    vids = []
    for name, filt in VARIANTS.items():
        reset() if filt is None else rotate(filt)
        v = gen(p, seed=SEED)
        export_to_video(list(v), f"results/zero_shot_rotation/p{i}_{name}.mp4", fps=8)
        vids.append(v)
    path = f"results/zero_shot_rotation/p{i}_compare.mp4"
    export_to_video(list(side_by_side(vids)), path, fps=8)
    print("  left -> right:", " | ".join(VARIANTS))
    display(Video(path, embed=True, width=768))

reset()
torch.cuda.empty_cache()

## 7. What to look for

Each comparison video is **normal | rot_all | rot_up**.

| Prompt | Direction flipped? (rot_all / rot_up) | Quality (good / artifacts / broken) | Notes |
|---|---|---|---|
| glass shattering | | | |
| person walking | | | |
| car driving right | | | |
| candle burning | | | |

- **Direction flipped:** glass reassembles, person walks backward, car goes left, candle grows.
- **Quality:** expect artifacts; the paper also saw poor frame quality without fine-tuning.
- **rot_all vs rot_up:** which gives the clearer reversal with fewer artifacts?

Whatever you find is a result: it's the "zero-shot rotation" experiment for the report.

## 8. (Optional) Try more seeds

In [ ]:
# One seed can mislead: re-run a prompt with a few seeds before concluding anything
p = prompts[0]
for seed in [1, 2]:
    vids = []
    for name, filt in VARIANTS.items():
        reset() if filt is None else rotate(filt)
        vids.append(gen(p, seed=seed))
    path = f"results/zero_shot_rotation/p0_seed{seed}_compare.mp4"
    export_to_video(list(side_by_side(vids)), path, fps=8)
    display(Video(path, embed=True, width=768))
reset()